# [DATASET_NAME] - Kaggle Data Science Project

---

## Project Overview

### Research Question
[What are you trying to predict or understand?]

### Objective
[What specific problem are we solving?]

### Dataset Information
- **Source**: Kaggle - [dataset link]
- **Sample Size**: [number of rows]
- **Features**: [number of columns]
- **Target Variable**: [what we are predicting]

## Setup and Environment

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["font.size"] = 10

In [ ]:
primary = "#351F7D"
secondary = "#C4A3CE"
accent_positive = "#27AE60"
accent_negative = "#E74C3C"
accent_neutral = "#3498DB"

## Data Loading and Initial Exploration

### Loading the Dataset

First look at what we're working with.

In [ ]:
file_path = "[DATASET_PATH]"
df = pd.read_csv(file_path)

### First Look

In [ ]:
print(f"Dataset shape: {df.shape}")
print(f"\nColumn names and types:\n{df.dtypes}")
print(f"\nFirst few rows:\n{df.head()}")

In [ ]:
print(f"Basic statistics:\n{df.describe()}")

### Data Quality Check

In [ ]:
print(f"Missing values:\n{df.isnull().sum()}")
print(f"\nDuplicate rows: {df.duplicated().sum()}")

## Exploratory Data Analysis (EDA)

### Separating Column Types

In [ ]:
numerical_cols = df.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = df.select_dtypes(include=["object"]).columns.tolist()

print(f"Numerical columns: {numerical_cols}")
print(f"\nCategorical columns: {categorical_cols}")

### Numerical Features Distribution

In [ ]:
for col in numerical_cols:
    print(f"\n{col}:")
    print(f"  Mean: {df[col].mean():.4f}")
    print(f"  Median: {df[col].median():.4f}")
    print(f"  Std Dev: {df[col].std():.4f}")
    print(f"  Min: {df[col].min():.4f}")
    print(f"  Max: {df[col].max():.4f}")

### Categorical Features Distribution

In [ ]:
for col in categorical_cols:
    print(f"\n{col}:")
    print(df[col].value_counts())

### Correlation Analysis

In [ ]:
correlation_matrix = df[numerical_cols].corr()
print("Correlation Matrix:")
print(correlation_matrix)

## Data Preprocessing

### Handling Missing Values

In [ ]:
missing_analysis = df.isnull().sum()
print(f"Columns with missing values:\n{missing_analysis[missing_analysis > 0]}")

# strategy: fill with mean, median, mode, or drop
# df["column_name"].fillna(df["column_name"].mean(), inplace=True)

### Removing Duplicates

In [ ]:
initial_rows = len(df)
df = df.drop_duplicates()
print(f"Rows removed: {initial_rows - len(df)}")

### Handling Outliers

In [ ]:
for col in numerical_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    outliers = df[(df[col] < lower_bound) | (df[col] > upper_bound)]
    if len(outliers) > 0:
        print(f"{col}: {len(outliers)} outliers detected")

### Encoding Categorical Variables

In [ ]:
from sklearn.preprocessing import LabelEncoder

# label encoding
for col in categorical_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))

## Feature Engineering

### Creating New Features

In [ ]:
# example: create interaction features
# df["feature_interaction"] = df["feature_1"] * df["feature_2"]

# example: create polynomial features
# df["feature_squared"] = df["feature_1"] ** 2

### Feature Selection

In [ ]:
target_col = "[TARGET_COLUMN]"

# select features by correlation with target
correlation_with_target = df.corr()[target_col].abs().sort_values(ascending=False)
print(f"Correlation with target:\n{correlation_with_target}")

## Train-Test Split

In [ ]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=[target_col])
y = df[target_col]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set: {X_train.shape[0]} rows")
print(f"Test set: {X_test.shape[0]} rows")

## Feature Scaling

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

X_train_scaled[numerical_cols] = scaler.fit_transform(X_train[numerical_cols])
X_test_scaled[numerical_cols] = scaler.transform(X_test[numerical_cols])

print("Features scaled successfully")

## Model Building

### Initialising Models

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

models = {
    "Logistic_Regression": LogisticRegression(random_state=42, max_iter=1000),
    "Decision_Tree": DecisionTreeClassifier(random_state=42),
    "Random_Forest": RandomForestClassifier(random_state=42, n_jobs=-1),
    "Gradient_Boosting": GradientBoostingClassifier(random_state=42),
}

### Training Models

In [ ]:
trained_models = {}

for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    trained_models[name] = model
    print(f"{name} trained")

### Cross-Validation

In [ ]:
from sklearn.model_selection import cross_val_score

cv_results = {}

for name, model in models.items():
    scores = cross_val_score(model, X_train_scaled, y_train, cv=5, scoring="accuracy")
    cv_results[name] = {
        "mean_score": scores.mean(),
        "std_score": scores.std()
    }
    print(f"{name}: {scores.mean():.4f} (+/- {scores.std():.4f})")

### Hyperparameter Tuning

In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    "n_estimators": [50, 100, 150],
    "max_depth": [5, 10, 15],
    "min_samples_split": [2, 5, 10],
}

grid_search = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)

grid_search.fit(X_train_scaled, y_train)
print(f"Best parameters: {grid_search.best_params_}")
print(f"Best score: {grid_search.best_score_:.4f}")

## Model Evaluation

### Generating Predictions

In [ ]:
predictions = {}

for name, model in trained_models.items():
    predictions[name] = model.predict(X_test_scaled)

### Performance Metrics

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

results = {}

for name, pred in predictions.items():
    results[name] = {
        "Accuracy": accuracy_score(y_test, pred),
        "Precision": precision_score(y_test, pred, average="weighted", zero_division=0),
        "Recall": recall_score(y_test, pred, average="weighted", zero_division=0),
        "F1_Score": f1_score(y_test, pred, average="weighted", zero_division=0),
    }

results_df = pd.DataFrame(results).T.sort_values("F1_Score", ascending=False)
print(results_df)

### Confusion Matrix

In [ ]:
from sklearn.metrics import confusion_matrix

best_model_name = results_df.index[0]
best_pred = predictions[best_model_name]
cm = confusion_matrix(y_test, best_pred)

print(f"Confusion Matrix for {best_model_name}:")
print(cm)

### Feature Importance

In [ ]:
best_model = trained_models[best_model_name]

if hasattr(best_model, "feature_importances_"):
    importances = best_model.feature_importances_
    importance_df = pd.DataFrame({
        "Feature": X_train_scaled.columns,
        "Importance": importances
    }).sort_values("Importance", ascending=False)
    
    print(f"Top 10 features from {best_model_name}:")
    print(importance_df.head(10))
else:
    print(f"{best_model_name} does not have feature importances")

### Saving the Best Model

In [ ]:
import pickle

model_filename = f"models/{best_model_name}.pkl"
with open(model_filename, "wb") as f:
    pickle.dump(best_model, f)

print(f"Model saved: {model_filename}")

## Summary and Findings

### Best Model Performance

[Summary of best model results]

In [ ]:
print(f"Best performing model: {best_model_name}")
print(f"\nPerformance metrics:")
print(results_df.iloc[0])

### Key Findings

- [Finding 1]
- [Finding 2]
- [Finding 3]

### Limitations and Next Steps

**Limitations:**
- [Limitation 1]
- [Limitation 2]

**Potential Improvements:**
- [Improvement 1]
- [Improvement 2]